# Notebook 50: Infinity-2B GGUF Gradio playground

This notebook launches the **unofficial Infinity-2B Q8_0 GGUF Gradio UI** in a fresh Google Colab runtime. It is a clean baseline playground: no PFB, SAC, style image, or custom style injection is used.

Use it to answer one question first: **does the GGUF model itself produce acceptable content images?** If the clean playground is poor, the problem is not caused by the style-transfer variants.

The notebook keeps the GGUF model and Infinity architecture from the Hugging Face repository, but replaces the UI's RAM-heavy T5 loading path with a streaming loader and places T5 on the GPU when CUDA is available.


In [ ]:
from pathlib import Path
import gc
import importlib.util
import math
import os
import re
import runpy
import shutil
import subprocess
import sys

# Colab paths and reproducible defaults.
ROOT = Path('/content/notebook_50_infinity2b_gguf_playground')
PORT_DIR = ROOT / 'gguf_port'
OFFICIAL_DIR = PORT_DIR / 'Infinity'
ASSET_DIR = ROOT / 'assets'
for path in (PORT_DIR, ASSET_DIR):
    path.mkdir(parents=True, exist_ok=True)

OFFICIAL_REPO = 'https://github.com/FoundationVision/Infinity.git'
GGUF_REPO = 'kzopp/Infinity-2B-GGUF_UNOFFICIAL'
DEFAULT_PN = '0.25M'  # 0.25M = 512x512; try 1M after the clean test works.
DEFAULT_CFG = 3.0
DEFAULT_TAU = 0.5
DEFAULT_SEED = 42
T5_DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

print('ROOT:', ROOT)
print('Defaults:', DEFAULT_PN, '| CFG:', DEFAULT_CFG, '| tau:', DEFAULT_TAU, '| seed:', DEFAULT_SEED)
print('T5 device:', T5_DEVICE)


In [ ]:
# Install the GGUF UI dependencies. Colab already supplies PyTorch/CUDA.
packages = [
    'gguf', 'gradio', 'transformers', 'sentencepiece',
    'easydict', 'typed-argument-parser', 'seaborn', 'kornia',
    'gputil', 'colorama', 'omegaconf', 'timm==0.9.6',
    'decord', 'pytz', 'imageio', 'einops', 'opencv-python', 'accelerate',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('Dependencies installed.')


In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA runtime is required for a practical Infinity-2B playground. In Colab select Runtime > Change runtime type > T4 GPU.')
props = torch.cuda.get_device_properties(0)
print('GPU:', props.name)
print('VRAM GiB:', round(props.total_memory / 2**30, 2))


In [ ]:
from huggingface_hub import hf_hub_download

if not OFFICIAL_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', OFFICIAL_REPO, str(OFFICIAL_DIR)], check=True)
else:
    print('Official Infinity source already exists:', OFFICIAL_DIR)

def download_hf_file(filename, target_dir):
    target_dir.mkdir(parents=True, exist_ok=True)
    return Path(hf_hub_download(repo_id=GGUF_REPO, filename=filename, local_dir=str(target_dir)))

PORT_SCRIPT = download_hf_file('generate_image_2b_q8_gguf.py', PORT_DIR)
PORT_UTILS = download_hf_file('infinity_gguf_utils.py', PORT_DIR)
UI_SCRIPT = download_hf_file('gradio_webui.py', PORT_DIR)
PATCHED_BASIC = download_hf_file('Infinity/infinity/models/basic.py', ROOT / 'gguf_patched_source')
PATCHED_INFINITY = download_hf_file('Infinity/infinity/models/infinity.py', ROOT / 'gguf_patched_source')
INFINITY_GGUF = download_hf_file('infinity_2b_reg_Q8_0.gguf', ASSET_DIR)
T5_GGUF = download_hf_file('flan-t5-xl-encoder-Q8_0.gguf', ASSET_DIR)
VAE_PATH = download_hf_file('Infinity/infinity_vae_d32_reg.pth', ASSET_DIR)

# Use the GGUF port's compatible Infinity modules.
shutil.copy2(PATCHED_BASIC, OFFICIAL_DIR / 'infinity' / 'models' / 'basic.py')
official_infinity = OFFICIAL_DIR / 'infinity' / 'models' / 'infinity.py'
shutil.copy2(PATCHED_INFINITY, official_infinity)

# The port may expose flash_attn_func=None; guard the constructor before importing it.
infinity_source = official_infinity.read_text()
old_guard = "customized_kernel_installed = any('Infinity' in arg_name for arg_name in flash_attn_func.__code__.co_varnames)"
new_guard = "customized_kernel_installed = flash_attn_func is not None and any('Infinity' in arg_name for arg_name in flash_attn_func.__code__.co_varnames)"
if old_guard in infinity_source:
    official_infinity.write_text(infinity_source.replace(old_guard, new_guard, 1))

for path in (PORT_SCRIPT, PORT_UTILS, UI_SCRIPT, INFINITY_GGUF, T5_GGUF, VAE_PATH):
    print(f'{path.name:40s} {path.stat().st_size / 2**30:.3f} GiB')
print('All playground files are ready.')


## Colab-safe T5 loading

The original Gradio launcher materializes a complete T5 state dictionary before constructing the model. That duplicates the encoder in host RAM. The function below loads one GGUF tensor at a time into an empty Flan-T5-XL model and keeps it on the GPU for a normal Colab T4 run.


In [ ]:
import numpy as np
import gguf

sys.path.insert(0, str(PORT_DIR))
sys.path.insert(0, str(OFFICIAL_DIR))

loader_spec = importlib.util.spec_from_file_location('generate_image_2b_q8_gguf', PORT_SCRIPT)
gguf_loader = importlib.util.module_from_spec(loader_spec)
sys.modules[loader_spec.name] = gguf_loader
loader_spec.loader.exec_module(gguf_loader)

def load_t5_encoder_streaming(gguf_path, device='cuda'):
    from accelerate import init_empty_weights
    from gguf import GGUFReader
    from transformers import T5Config, T5EncoderModel

    key_map = {
        'enc.': 'encoder.', '.blk.': '.block.', 'token_embd': 'shared',
        'output_norm': 'final_layer_norm',
        'attn_q': 'layer.0.SelfAttention.q', 'attn_k': 'layer.0.SelfAttention.k',
        'attn_v': 'layer.0.SelfAttention.v', 'attn_o': 'layer.0.SelfAttention.o',
        'attn_norm': 'layer.0.layer_norm',
        'attn_rel_b': 'layer.0.SelfAttention.relative_attention_bias',
        'ffn_up': 'layer.1.DenseReluDense.wi_1',
        'ffn_down': 'layer.1.DenseReluDense.wo',
        'ffn_gate': 'layer.1.DenseReluDense.wi_0',
        'ffn_norm': 'layer.1.layer_norm',
    }
    config = T5Config.from_pretrained('google/flan-t5-xl')
    with init_empty_weights():
        model = T5EncoderModel(config)
    model = model.to(dtype=torch.float16)
    model = model.to_empty(device=device)
    model.eval()
    model.requires_grad_(False)

    parameter_refs = dict(model.named_parameters())
    buffer_refs = dict(model.named_buffers())
    reader = GGUFReader(str(gguf_path))
    plain_types = {gguf.GGMLQuantizationType.F32, gguf.GGMLQuantizationType.F16}
    loaded = 0
    skipped = []
    with torch.inference_mode():
        for tensor in reader.tensors:
            name = tensor.name
            for old_key, new_key in key_map.items():
                name = name.replace(old_key, new_key)
            shape = torch.Size(tuple(int(v) for v in reversed(tensor.shape)))
            raw = torch.from_numpy(np.array(tensor.data))
            if tensor.tensor_type not in plain_types:
                param = gguf_loader.GGUFParameter(raw, quant_type=tensor.tensor_type)
                value = gguf_loader.dequantize_gguf_tensor(param, target_dtype=torch.float16)
            else:
                value = raw.to(dtype=torch.float16)
            if value.numel() != math.prod(shape):
                skipped.append((name, 'numel mismatch'))
                del raw, value
                continue
            value = value.reshape(shape)
            target = parameter_refs.get(name)
            if target is None:
                target = buffer_refs.get(name)
            if target is None or tuple(target.shape) != tuple(shape):
                skipped.append((name, 'missing or shape mismatch'))
                del raw, value
                continue
            target.data.copy_(value.to(device=target.device, dtype=target.dtype))
            loaded += 1
            del raw, value
    del reader, parameter_refs, buffer_refs
    gc.collect()
    model.eval()
    model.requires_grad_(False)
    print(f'Streaming T5 loaded on {device}: {loaded} tensors, skipped {len(skipped)}')
    if skipped:
        print('First skipped tensors:', skipped[:5])
    return model

def colab_t5_loader(gguf_path, device='cpu'):
    # The UI requests CPU, but GPU placement avoids the host-RAM spike in free Colab.
    return load_t5_encoder_streaming(gguf_path, device=T5_DEVICE)

# The Gradio UI imports this function from the already-loaded module.
gguf_loader.load_t5_encoder_from_gguf = colab_t5_loader
print('GGUF loader imported and Colab-safe T5 loader installed.')


In [ ]:
# Confirm the patched module can be imported before launching the UI.
assert hasattr(gguf_loader, 'load_infinity_from_gguf')
assert hasattr(gguf_loader, 'generate_image')
assert INFINITY_GGUF.exists() and T5_GGUF.exists() and VAE_PATH.exists()
print('Loader smoke check passed.')


## Launch the clean GGUF playground

Run the next cell and wait for the public Gradio URL. In the UI:

1. Set **Resolution Preset** to `0.25M` for the first test.
2. Click **Load Models** once.
3. Use CFG `3.0`, tau `0.5`, seed `42`, and disable random seed.
4. Generate simple prompts without style words.
5. Only after the baseline looks reasonable, try `1M` in a fresh runtime.

The UI's GGUF generation script uses its own fixed sampling defaults for top-k/top-p; this notebook is intended to validate the model's clean baseline, not to compare custom PFB/SAC code.


In [ ]:
# Launch the repository's Gradio UI in this same process so the patched T5 loader is used.
os.chdir(PORT_DIR)
sys.argv = [
    str(UI_SCRIPT),
    '--share',
    '--server-name', '0.0.0.0',
    '--infinity-gguf', str(INFINITY_GGUF),
    '--t5-gguf', str(T5_GGUF),
    '--vae-path', str(VAE_PATH),
]
runpy.run_path(str(UI_SCRIPT), run_name='__main__')


## Clean-baseline test prompts

Use these one at a time in the playground:

```text
a red apple on a wooden table
a lighthouse beside the sea
a blue bicycle beside a brick wall
a mountain lake beneath a bright sky
a steam train traveling through green countryside
a vintage camera on a desk
a stone castle beside a lake
a colorful hot air balloon above a valley
```

For a fair diagnosis, keep the prompt, seed, CFG, tau, and resolution identical when comparing this playground against Notebook 9. Do not judge the style-transfer method until the baseline content image is acceptable.
